![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 9. Déployer avec Streamlit Community Cloud</b>

# 0. De votre machine au monde entier

Dans le notebook précédent, vous avez construit une application Streamlit complète — avec ses pages, ses filtres, ses graphiques et ses exports. Vous l'avez testée en local avec `streamlit run app.py`, et tout fonctionne. Mais pour l'instant, votre application ne vit que sur votre ordinateur. Si vous fermez le terminal, elle disparaît.

Dans ce notebook, nous allons franchir la dernière étape : **rendre votre application accessible à n'importe qui, depuis un simple lien web**. C'est ce qu'on appelle le déploiement.

## 0.1. Pourquoi déployer ?

Quand vous lancez `streamlit run app.py`, Streamlit crée un petit serveur web local sur votre machine. Vous y accédez via `localhost:8501`, mais personne d'autre ne peut y accéder. Déployer, c'est simplement installer votre application sur un serveur distant qui tourne en permanence, et qui est accessible depuis n'importe quel navigateur.

Concrètement, déployer permet de partager votre travail avec vos collègues ou clients sans qu'ils aient besoin d'installer Python, ni de comprendre le code. Ils cliquent sur un lien, et votre dashboard s'affiche.

## 0.2. Ce que nous allons couvrir

Il existe plusieurs façons de déployer une application Streamlit : Docker, des serveurs cloud (AWS, Azure, GCP…), des plateformes spécialisées, etc. Dans ce notebook, nous allons nous concentrer sur la méthode la plus simple et la plus rapide : **[Streamlit Community Cloud](https://streamlit.io/cloud)**. C'est une plateforme gratuite, maintenue par l'équipe Streamlit elle-même, qui vous permet de mettre en ligne votre application en quelques clics à partir d'un dépôt GitHub.

Nous verrons aussi comment gérer proprement les **secrets** (clés API, mots de passe…) pour ne jamais les exposer dans votre code.

💡 Le déploiement via Docker et les pipelines CI/CD seront abordés dans la masterclass.

À la fin de ce notebook, vous saurez mettre en ligne votre application Streamlit et la partager via un lien public. Vous êtes à la dernière étape de la formation — bravo pour le chemin parcouru jusqu'ici !

# 1. Secrets et configuration : ne jamais mettre vos mots de passe dans le code

Avant de déployer, il y a un réflexe important à prendre. Quand vous développez une application, il arrive souvent d'avoir besoin d'informations sensibles : une clé d'API pour interroger un service externe, un mot de passe de base de données, un token d'authentification… Ces informations s'appellent des **secrets**.

⚠️ Ne mettez **jamais** vos secrets directement dans votre code Python ou vos notebooks. Si vous poussez votre code sur GitHub, ces informations deviennent publiques et accessibles à tous. C'est l'une des erreurs les plus courantes — et les plus dangereuses — en développement.

## 1.1. Pourquoi séparer les secrets du code ?

L'idée est simple : vos secrets vivent dans un fichier à part (ou dans une interface de configuration), et votre code va les lire au moment de l'exécution. Cela vous apporte trois avantages concrets. D'abord, la **sécurité** : vos identifiants ne sont jamais visibles dans votre dépôt Git. Ensuite, la **flexibilité** : vous pouvez changer un mot de passe sans toucher au code. Enfin, c'est une **bonne pratique** qui facilite la collaboration — chaque développeur ou environnement (local, production…) peut avoir ses propres secrets sans conflit.

## 1.2. Les méthodes de gestion des secrets

Voyons les principales approches, de la plus spécifique à Streamlit à la plus générale.

### `st.secrets` — la méthode native de Streamlit

Streamlit propose son propre système de secrets. En local, vous créez un fichier `.streamlit/secrets.toml` dans votre projet. Ce fichier contient vos secrets au format [TOML](https://toml.io/). Lors du déploiement sur Streamlit Community Cloud, vous renseignez ces mêmes valeurs directement dans l'interface web — le fichier n'a donc jamais besoin d'être versionné.

```python
import streamlit as st

# Lire un secret depuis le fichier secrets.toml
token = st.secrets["api"]["token"]
st.write("Token chargé :", bool(token))
```

⚠️ Pensez à ajouter `.streamlit/secrets.toml` à votre fichier `.gitignore` pour qu'il ne soit jamais poussé sur GitHub.

### `st.query_params` — passer des informations via l'URL

Parfois, vous voulez transmettre un paramètre de configuration directement dans l'URL de votre application (par exemple, un identifiant de vue ou un filtre par défaut). Streamlit permet de lire et écrire ces paramètres avec `st.query_params`.

```python
import streamlit as st

# Lire les paramètres de l'URL (ex: ?ville=Paris)
params = st.query_params
ville = params.get("ville", "Non spécifiée")
st.write(f"Ville sélectionnée : {ville}")
```

💡 Les query params ne sont pas faits pour stocker des secrets — ils sont visibles dans la barre d'adresse du navigateur.

### Variables d'environnement — l'approche classique

En dehors de Streamlit, la méthode la plus répandue consiste à utiliser des **variables d'environnement**. C'est particulièrement utile avec Docker ou des pipelines CI/CD.

```python
import os

token = os.environ.get("API_TOKEN")
```

### Fichiers `.env` ou `.toml`

Vous pouvez aussi stocker vos secrets dans un fichier dédié (`.env`, `.toml`…) et le charger avec une bibliothèque comme `python-dotenv`. L'essentiel est que ce fichier ne soit **jamais versionné**.

📖 Documentation complète : [Secrets management — Streamlit Docs](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/secrets-management)

In [ ]:
%%writefile streamlit_app/.streamlit/secrets.toml.example
# Exemple de fichier de configuration des secrets pour Streamlit.
# Ce fichier NE DOIT PAS être versionné en production (utilisez .gitignore).
# Renommez ce fichier en 'secrets.toml' et placez vos vrais secrets ici.

[api]
token = "VOTRE_JETON_ICI"  # Remplacez par votre vrai token d'API


In [ ]:
# Exemple d'accès aux secrets
# import streamlit as st
# token = st.secrets.get("api", {}).get("token", None)
# st.write("Token configuré ?", bool(token))


# 2. Déployer avec Streamlit Community Cloud

Passons à la pratique. [Streamlit Community Cloud](https://streamlit.io/cloud) est une plateforme gratuite maintenue par Streamlit qui vous permet de mettre votre application en ligne en quelques minutes. Vous n'avez aucun serveur à configurer, aucune infrastructure à gérer — il suffit d'un dépôt GitHub et de quelques clics.

## 2.1. Pourquoi Streamlit Community Cloud ?

Streamlit Community Cloud est la solution la plus directe pour déployer une application Streamlit. Vous poussez votre code sur GitHub, vous connectez le dépôt à la plateforme, et votre application est en ligne. À chaque fois que vous modifiez votre code sur GitHub, l'application se met à jour automatiquement — vous n'avez rien d'autre à faire.

La plateforme gère aussi les secrets via une interface dédiée (ce que nous avons vu dans la section précédente avec `st.secrets`), et vous obtenez un lien public que vous pouvez partager directement.

💡 C'est la solution idéale pour un prototype, une démonstration, un portfolio de projets, ou tout simplement pour montrer votre travail à un recruteur ou un client.

## 2.2. Les étapes du déploiement

Le processus se fait en quatre temps.

**1. Préparez votre dépôt GitHub.** Votre projet doit être hébergé sur un dépôt GitHub (public ou privé). Assurez-vous que votre fichier principal s'appelle `app.py` (ou que le chemin est clair), et que toutes vos dépendances sont listées dans un fichier `requirements.txt` à la racine du projet. Ce fichier est indispensable : c'est lui qui indique à Streamlit Cloud quelles bibliothèques installer.

**2. Connectez votre dépôt à Streamlit Cloud.** Rendez-vous sur [streamlit.io/cloud](https://streamlit.io/cloud) et connectez-vous avec votre compte GitHub. Cliquez sur **"New app"**, sélectionnez votre dépôt, la branche (en général `main`) et le chemin vers votre fichier principal.

**3. Configurez vos secrets.** Si votre application utilise des secrets (clés API, tokens…), renseignez-les dans l'onglet **"Secrets"** de l'interface Streamlit Cloud. Ces valeurs seront accessibles dans votre code via `st.secrets`, exactement comme en local avec le fichier `secrets.toml`.

**4. Déployez.** Cliquez sur **"Deploy"**. En quelques secondes, votre application est en ligne et accessible via une URL publique que vous pouvez partager immédiatement.

⚠️ Pensez à vérifier que votre `requirements.txt` est à jour avant de déployer. Si une bibliothèque manque, le déploiement échouera et Streamlit Cloud vous affichera un message d'erreur dans les logs.

# <font color='#ff7373'><b>Félicitations !</b></font>

Vous avez terminé l'ensemble de la formation ! En partant d'un simple fichier `app.py`, vous êtes désormais capables de construire des interfaces web interactives, de les structurer en plusieurs pages, de gérer l'état et les interactions utilisateur, et maintenant de les mettre en ligne pour les partager avec le monde.

Streamlit Community Cloud est un excellent point de départ pour vos déploiements. Pour aller plus loin — conteneurisation avec Docker, automatisation avec GitHub Actions, déploiement sur des infrastructures cloud — nous approfondirons tout cela dans la masterclass.

📖 Pour continuer à explorer : [Documentation Streamlit](https://docs.streamlit.io/) · [Streamlit Community Cloud](https://streamlit.io/cloud) · [Gallery d'applications](https://streamlit.io/gallery)

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>